In [2]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())


torch: 2.10.0+cpu
cuda available: False


In [3]:
OUT_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/output")
parquet_path = OUT_DIR / "ait_ads_wazuh.parquet"

df_wazuh = pd.read_parquet(parquet_path)
print("Loaded df_wazuh:", df_wazuh.shape)

def map_priority(level):
    if pd.isna(level):
        return None
    level = float(level)
    if level <= 3:
        return "low"
    elif level <= 6:
        return "medium"
    elif level <= 10:
        return "high"
    else:
        return "critical"

df = df_wazuh.copy()
df["y_priority"] = df["rule_level"].apply(map_priority)

df_ml = df.dropna(subset=["full_log", "y_priority"]).copy()
df_ml["full_log"] = df_ml["full_log"].astype("string").fillna("")
print("After dropna:", df_ml.shape)
display(df_ml["y_priority"].value_counts())


Loaded df_wazuh: (2600263, 14)
After dropna: (2293628, 15)


y_priority
medium      1873973
low          286087
high         131901
critical       1667
Name: count, dtype: int64

In [ ]:
RANDOM_STATE = 42


PER_CLASS = 10_000

parts = []
for cls in ["critical", "high", "low", "medium"]:
    df_c = df_ml[df_ml["y_priority"] == cls]
    n = len(df_c)
    if cls == "critical":
        # берём все, иначе критикал снова исчезнет
        parts.append(df_c)
        print(cls, "take ALL:", n)
    else:
        take_n = min(PER_CLASS, n)
        parts.append(df_c.sample(n=take_n, random_state=RANDOM_STATE))
        print(cls, "take:", take_n, "out of", n)

df_bal = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
print("Balanced df:", df_bal.shape)
display(df_bal["y_priority"].value_counts())


critical take ALL: 1667
high take: 10000 out of 131901
low take: 10000 out of 286087
medium take: 10000 out of 1873973
Balanced df: (31667, 15)


y_priority
medium      10000
high        10000
low         10000
critical     1667
Name: count, dtype: int64

In [5]:
X = df_bal["full_log"].tolist()
y_text = df_bal["y_priority"].tolist()

le = LabelEncoder()
y = le.fit_transform(y_text)
print("Classes:", list(le.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", len(X_train), "Test:", len(X_test))
print("Test class counts:", np.bincount(y_test))


Classes: [np.str_('critical'), np.str_('high'), np.str_('low'), np.str_('medium')]
Train: 25333 Test: 6334
Test class counts: [ 334 2000 2000 2000]


In [6]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_dict({"text": X_train, "label": y_train})
test_ds  = Dataset.from_dict({"text": X_test,  "label": y_test})

def tok_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

train_ds = train_ds.map(tok_fn, batched=True, remove_columns=["text"])
test_ds  = test_ds.map(tok_fn, batched=True, remove_columns=["text"])

train_ds.set_format("torch")
test_ds.set_format("torch")

train_ds[0].keys()


/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 6334/6334 [00:00<00:00, 21000.17 examples/s]


dict_keys(['label', 'input_ids', 'token_type_ids', 'attention_mask'])

In [8]:
import inspect
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

num_labels = len(le.classes_)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    acc = (preds == labels).mean()
    return {"accuracy": acc, "macro_f1": macro_f1}

# --- делаем kwargs совместимыми с разными версиями transformers ---
sig = inspect.signature(TrainingArguments.__init__).parameters

kwargs = dict(
    output_dir="hf_out",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
)

# названия стратегий отличаются в версиях
if "evaluation_strategy" in sig:
    kwargs["evaluation_strategy"] = "epoch"
elif "eval_strategy" in sig:
    kwargs["eval_strategy"] = "epoch"

if "save_strategy" in sig:
    kwargs["save_strategy"] = "epoch"

# лучшие веса в конце (если поддерживается)
if "load_best_model_at_end" in sig:
    kwargs["load_best_model_at_end"] = True
if "metric_for_best_model" in sig:
    kwargs["metric_for_best_model"] = "macro_f1"
if "greater_is_better" in sig:
    kwargs["greater_is_better"] = True

# отключаем wandb/прочие репорты (если поддерживается)
if "report_to" in sig:
    kwargs["report_to"] = "none"

# fp16 на CPU не нужен (если поддерживается)
if "fp16" in sig:
    kwargs["fp16"] = False

training_args = TrainingArguments(**kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

print("TrainingArguments OK. Using keys:", list(kwargs.keys()))


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 691.53it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TrainingArguments OK. Using keys: ['output_dir', 'per_device_train_batch_size', 'per_device_eval_batch_size', 'num_train_epochs', 'learning_rate', 'weight_decay', 'logging_steps', 'eval_strategy', 'save_strategy', 'load_best_model_at_end', 'metric_for_best_model', 'greater_is_better', 'report_to', 'fp16']


In [9]:
trainer.train()


/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.562556,0.523927,0.644616,0.548229
2,0.500324,0.521505,0.673982,0.606666
3,0.512071,0.516127,0.676192,0.611532


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.54it/s]
/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.25it/s]
/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.69it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma']

TrainOutput(global_step=9501, training_loss=0.5206967952966364, metrics={'train_runtime': 9482.3098, 'train_samples_per_second': 8.015, 'train_steps_per_second': 1.002, 'total_flos': 2516937226417152.0, 'train_loss': 0.5206967952966364, 'epoch': 3.0})

In [10]:
pred = trainer.predict(test_ds)
logits = pred.predictions
y_true = pred.label_ids
y_pred = np.argmax(logits, axis=1)

print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

    critical       0.51      0.74      0.60       334
        high       0.53      0.91      0.67      2000
         low       1.00      1.00      1.00      2000
      medium       0.56      0.10      0.18      2000

    accuracy                           0.68      6334
   macro avg       0.65      0.69      0.61      6334
weighted avg       0.69      0.68      0.61      6334

Confusion matrix:
 [[ 248    0    0   86]
 [  95 1827    1   77]
 [   0    0 2000    0]
 [ 145 1647    0  208]]
